# Primary Care visits Data generation

Freqüència d'interacció amb el sistema de salut durant el període d'estudi. 

- Visites a Atenció Primària: Mitjana de 6,3 visites/persona (o 5,36 visites/any). 

- Visites a Salut Mental Especialitzada: Mitjana de 0,78 visites. 

- Urgències: Mitjana de 0,43 visites per motius psiquiàtrics i 0,51 per motius mèdics generals. 

Hospitalitzacions: 

- Data d'ingrés i data d'alta. 

- Dies d'estada en unitats psiquiàtriques (Mitjana: 0,37 dies/any). 

- Motiu de l'ingrés i diagnòstic principal de l'alta. 

### Part 1: Relacionar id amb el dataset original

In [2]:
import pandas as pd

# 1. Llegim el teu fitxer original
path_base = r'C:\Users\LRAJAG\Desktop\Data_sintetica\csv\1_Sociodemographic\pacients_100k_sintetic.csv'
df_pacients = pd.read_csv(path_base)

# 2. Agafem la columna d'ID (sigui quin sigui el seu nom)
col_id = [c for c in df_pacients.columns if 'id' in c.lower()][0]
llista_ids = df_pacients[col_id].tolist()

print(f"Tenim {len(llista_ids)} pacients preparats per rebre visites.")

Tenim 100000 pacients preparats per rebre visites.


### Part 2: Freqüència tipus de visites

In [3]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

# Mitjanes de visites PER ANY 
config_visites = {
    'Atenció Primària': 5.36,
    'Salut Mental Especialitzada': 0.78,
    'Urgències Psiquiàtriques': 0.43,
    'Urgències Mèdiques': 0.51,
    'Hospitalització': 0.02 
}

anys_estudi = 7  # Del 2011 al 2017
data_inici = datetime(2011, 1, 1)

### Part 3: Generar les visites

In [4]:
registres = []
MAX_ENTRADES = 100000  

# Calculem les probabilitats relatives segons les mitjanes
tipus_opcions = list(config_visites.keys())
pesos = [config_visites[t] for t in tipus_opcions]
probabilitats = [p / sum(pesos) for p in pesos]

for i in range(MAX_ENTRADES):
    # Mostrem el progrés
    if (i + 1) % 20000 == 0:
        print(f"Generades {i + 1} entrades...")

    # Triem un pacient i un tipus de visita a l'atzar
    pacient = random.choice(llista_ids)
    tipus = np.random.choice(tipus_opcions, p=probabilitats)
    
    # Generem la data
    dies_aleatoris = random.randint(0, 2555)
    data_visita = data_inici + timedelta(days=dies_aleatoris)
    
    registres.append({
        'id_pacient': pacient,
        'tipus_visita': tipus,
        'data_ingres': data_visita.strftime('%Y-%m-%d')
    })

Generades 20000 entrades...
Generades 40000 entrades...
Generades 60000 entrades...
Generades 80000 entrades...
Generades 100000 entrades...


### Part 4: Generar csv

In [5]:
print("Convertint dades a format taula (això pot trigar un moment)...")
df_visites = pd.DataFrame(registres)

# Ordenem per data per a que sigui cronològic
df_visites.sort_values(by='data_ingres', inplace=True)

path_sortida = r'C:\Users\LRAJAG\Desktop\Data_sintetica\csv\4_Primarycare_visits\Registre_Visites_TOTAL_100k.csv'
df_visites.to_csv(path_sortida, index=False)

print(f"S'han generat un total de {len(df_visites):,} visites.")

Convertint dades a format taula (això pot trigar un moment)...
S'han generat un total de 100,000 visites.


### Part 5: Afegir dies d'estada

In [6]:
import pandas as pd
import numpy as np

# 1. Carreguem el fitxer
path_visites = r'C:\Users\LRAJAG\Desktop\Data_sintetica\csv\4_Primarycare_visits\Registre_Visites_TOTAL_100k.csv'
df = pd.read_csv(path_visites)

print("Processant dates i estades...")

# --- PAS CRÍTIC: Convertim el text a format DATA real ---
df['data_ingres'] = pd.to_datetime(df['data_ingres'])

# 2. Definim la funció per calcular l'estada
def calcular_estada(fila):
    if fila['tipus_visita'] == 'Hospitalització':
        # Generem una estada realista (mitjana de 18 dies)
        dies = max(1, int(np.random.gamma(shape=2, scale=9)))
        return dies
    else:
        return 0

# Apliquem la funció per crear la columna numèrica
df['dies_estada'] = df.apply(calcular_estada, axis=1)

# 3. Calculem la Data d'Alta (Suma matemàtica de dates)
df['data_alta'] = df['data_ingres'] + pd.to_timedelta(df['dies_estada'], unit='D')

# 4. PAS FINAL: Passem les dates a format text (YYYY-MM-DD) per al CSV
df['data_ingres'] = df['data_ingres'].dt.strftime('%Y-%m-%d')
df['data_alta'] = df['data_alta'].dt.strftime('%Y-%m-%d')

# 5. Guardem el fitxer actualitzat
df.to_csv(path_visites, index=False)

print(df[df['tipus_visita'] == 'Hospitalització'].head())

Processant dates i estades...
      id_pacient     tipus_visita data_ingres  dies_estada   data_alta
626        55501  Hospitalització  2011-01-17           10  2011-01-27
896        11030  Hospitalització  2011-01-23           27  2011-02-19
1544       95649  Hospitalització  2011-02-09           10  2011-02-19
1865       75072  Hospitalització  2011-02-17            3  2011-02-20
1964       87438  Hospitalització  2011-02-20            8  2011-02-28


In [7]:
df.tipus_visita.unique()

array(['Atenció Primària', 'Salut Mental Especialitzada',
       'Urgències Mèdiques', 'Urgències Psiquiàtriques',
       'Hospitalització'], dtype=object)

In [8]:
df.dies_estada.unique()

array([ 0, 10, 27,  3,  8, 31, 39,  4, 26, 20,  9, 47, 14, 12,  5,  2, 17,
       16, 46, 19, 58, 52, 13,  7,  6,  1, 29, 22, 36, 25, 15, 38, 37, 18,
       11, 64, 53, 21, 59, 34, 84, 24, 48, 23, 33, 28, 55])